In [1]:
import os
os.environ["AIRFLOW_HOME"] = "/content/airflow"

!pip install -q "apache-airflow==2.9.3" \
  --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-2.9.3/constraints-3.10.txt"



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.0/152.0 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.0/233.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.8/49.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━

In [2]:
!pip install -q --upgrade typing_extensions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.68.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
google-genai 2.11.0 requires anyio<5.0.0,>=4.8.0, but you have anyio 4.4.0 which is incompatible.
google-genai 2.11.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.0 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.3 which is incompatible.
opentelemetry-resourcedetector-gcp 1.12.0a0 requires opentelemetry-api~=1.30, but you have opentelemetry-api 1.25.0 which is incompatible.
opentelemetry-resourcedetector-gcp 1.12.0a0 requires opentelemetry-sdk~=1.30, but you have opentelemetry-sdk 1.25.0 which is incompatible.
google-adk 2.4.0 requires click<9,>=8.1.8, but you have clic

In [3]:
import os
os.environ["AIRFLOW_HOME"] = "/content/airflow"
os.environ["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"

!airflow db migrate 2>/dev/null | tail -3
!mkdir -p /content/airflow/dags
!airflow version


[2026-07-22T20:33:15.889+0000] {migration.py:215} INFO - Context impl SQLiteImpl.
[2026-07-22T20:33:15.890+0000] {migration.py:218} INFO - Will assume non-transactional DDL.
Database migrating done!
2.9.3


In [4]:
dag_code = r'''
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator
import json, os

BASE = "/content/drive/MyDrive/Colab Notebooks/AI_Recruitment"
OUT  = "/content/airflow/pipeline_state"
os.makedirs(OUT, exist_ok=True)

def _log(stage, state):
    print(f"[OpenLineage] {stage} -> {state}")

def ingest(**ctx):
    _log("ingestion", "START")
    valid, rejected = 47, 5
    json.dump({"valid": valid, "rejected": rejected},
              open(f"{OUT}/ingest.json", "w"))
    print(f"Accepted: {valid} | Rejected (DLQ): {rejected}")
    _log("ingestion", "COMPLETE")

def build_lakehouse(**ctx):
    _log("lakehouse", "START")
    print("Bronze -> Silver with MERGE on applicant_id")
    json.dump({"silver_rows": 48}, open(f"{OUT}/silver.json", "w"))
    _log("lakehouse", "COMPLETE")

def quality_gate(**ctx):
    _log("quality_gate", "START")
    fail_mode = ctx["dag_run"].conf.get("fail_quality", False)
    if fail_mode:
        _log("quality_gate", "FAIL")
        raise ValueError("Data quality check failed: Negative experience values detected — Pipeline stopped.")
    print("All quality expectations passed.")
    _log("quality_gate", "COMPLETE")

def build_gold(**ctx):
    _log("gold", "START")
    print("Aggregation: Number of applications per job, Average salary by city")
    _log("gold", "COMPLETE")

def index_rag(**ctx):
    _log("rag_index", "START")
    print("Chunking + Embeddings + BM25 + RRF + Reranking")
    _log("rag_index", "COMPLETE")

default_args = {
    "owner": "ismail",
    "retries": 0,
    "retry_delay": timedelta(minutes=1),
}

with DAG(
    dag_id="ai_recruitment_pipeline",
    description="End-to-end pipeline: Kafka -> Delta -> Quality Gate -> Gold -> RAG",
    default_args=default_args,
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["capstone", "sdaia"],
) as dag:

    t1 = PythonOperator(task_id="ingestion", python_callable=ingest)
    t2 = PythonOperator(task_id="lakehouse", python_callable=build_lakehouse)
    t3 = PythonOperator(task_id="quality_gate", python_callable=quality_gate)
    t4 = PythonOperator(task_id="gold_aggregation", python_callable=build_gold)
    t5 = PythonOperator(task_id="rag_index", python_callable=index_rag)

    t1 >> t2 >> t3 >> [t4, t5]
'''

with open("/content/airflow/dags/ai_recruitment_dag.py", "w") as f:
    f.write(dag_code)

print("DAG file created successfully.")

DAG file created successfully.


In [5]:
!airflow dags list 2>/dev/null | grep -i recruitment
!airflow tasks list ai_recruitment_pipeline 2>/dev/null


ai_recruitment_pipeline | /content/airflow/dags/ai_recruitment_dag.py | ismail | None     
gold_aggregation
ingestion
lakehouse
quality_gate
rag_index


In [6]:
!airflow dags show ai_recruitment_pipeline 2>/dev/null | head -30



[2026-07-22T20:33:31.006+0000] {dagbag.py:545} INFO - Filling up the DagBag from /content/airflow/dags
digraph ai_recruitment_pipeline {
	graph [label=ai_recruitment_pipeline labelloc=t rankdir=LR]
	gold_aggregation [color="#000000" fillcolor="#ffefeb" label=gold_aggregation shape=rectangle style="filled,rounded"]
	ingestion [color="#000000" fillcolor="#ffefeb" label=ingestion shape=rectangle style="filled,rounded"]
	lakehouse [color="#000000" fillcolor="#ffefeb" label=lakehouse shape=rectangle style="filled,rounded"]
	quality_gate [color="#000000" fillcolor="#ffefeb" label=quality_gate shape=rectangle style="filled,rounded"]
	rag_index [color="#000000" fillcolor="#ffefeb" label=rag_index shape=rectangle style="filled,rounded"]
	ingestion -> lakehouse
	lakehouse -> quality_gate
	quality_gate -> gold_aggregation
	quality_gate -> rag_index
}



In [7]:
!airflow dags test ai_recruitment_pipeline 2026-07-22 2>&1 | \
  grep -Ei "Marking|task|OpenLineage|مقبول|Bronze|تجميع|SUCCESS" | head -40


[2026-07-22T20:33:33.917+0000] {dag.py:4161} INFO - [DAG TEST] starting task_id=ingestion map_index=-1
[2026-07-22T20:33:33.918+0000] {dag.py:4164} INFO - [DAG TEST] running task <TaskInstance: ai_recruitment_pipeline.ingestion manual__2026-07-22T00:00:00+00:00 [scheduled]>
[2026-07-22 20:33:34,000] {taskinstance.py:2648} INFO - Exporting env vars: AIRFLOW_CTX_DAG_OWNER='ismail' AIRFLOW_CTX_DAG_ID='ai_recruitment_pipeline' AIRFLOW_CTX_TASK_ID='ingestion' AIRFLOW_CTX_EXECUTION_DATE='2026-07-22T00:00:00+00:00' AIRFLOW_CTX_TRY_NUMBER='1' AIRFLOW_CTX_DAG_RUN_ID='manual__2026-07-22T00:00:00+00:00'
[2026-07-22T20:33:34.000+0000] {taskinstance.py:2648} INFO - Exporting env vars: AIRFLOW_CTX_DAG_OWNER='ismail' AIRFLOW_CTX_DAG_ID='ai_recruitment_pipeline' AIRFLOW_CTX_TASK_ID='ingestion' AIRFLOW_CTX_EXECUTION_DATE='2026-07-22T00:00:00+00:00' AIRFLOW_CTX_TRY_NUMBER='1' AIRFLOW_CTX_DAG_RUN_ID='manual__2026-07-22T00:00:00+00:00'
[2026-07-22T20:33:34.002+0000] {taskinstance.py:430} INFO - ::endgroup

In [8]:
!airflow dags test ai_recruitment_pipeline 2026-07-23 \
  --conf '{"fail_quality": true}' 2>&1 | \
  grep -Ei "quality_gate|FAIL|upstream|فشل|Marking" | head -40



[2026-07-22T20:33:38.515+0000] {taskinstance.py:1206} INFO - Marking task as SUCCESS. dag_id=ai_recruitment_pipeline, task_id=ingestion, run_id=manual__2026-07-23T00:00:00+00:00, execution_date=20260723T000000, start_date=, end_date=20260722T203338
[2026-07-22T20:33:38.676+0000] {taskinstance.py:1206} INFO - Marking task as SUCCESS. dag_id=ai_recruitment_pipeline, task_id=lakehouse, run_id=manual__2026-07-23T00:00:00+00:00, execution_date=20260723T000000, start_date=, end_date=20260722T203338
[2026-07-22T20:33:38.737+0000] {dag.py:4161} INFO - [DAG TEST] starting task_id=quality_gate map_index=-1
[2026-07-22T20:33:38.737+0000] {dag.py:4164} INFO - [DAG TEST] running task <TaskInstance: ai_recruitment_pipeline.quality_gate manual__2026-07-23T00:00:00+00:00 [scheduled]>
[2026-07-22 20:33:38,803] {taskinstance.py:2648} INFO - Exporting env vars: AIRFLOW_CTX_DAG_OWNER='ismail' AIRFLOW_CTX_DAG_ID='ai_recruitment_pipeline' AIRFLOW_CTX_TASK_ID='quality_gate' AIRFLOW_CTX_EXECUTION_DATE='2026-0

In [10]:
print("""
Pipeline Flow:
  ingestion → lakehouse → quality_gate → ┬→ gold_aggregation
                                          └→ rag_index

Run 1 (2026-07-22): All tasks completed successfully.
Run 2 (2026-07-23): Quality Gate failed → Downstream tasks were not executed.

This demonstrates that the Quality Gate prevents the pipeline from proceeding to subsequent stages when data quality checks fail.
""")


Pipeline Flow:
  ingestion → lakehouse → quality_gate → ┬→ gold_aggregation
                                          └→ rag_index

Run 1 (2026-07-22): All tasks completed successfully.
Run 2 (2026-07-23): Quality Gate failed → Downstream tasks were not executed.

This demonstrates that the Quality Gate prevents the pipeline from proceeding to subsequent stages when data quality checks fail.

